In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

In [ ]:
class RobotVisionSystem:
    def __init__(self,model_path="yolov8n.pt"):  #initializing the modelька
        self.model=YOLO(model_path)  #the modelька itself
        self.target_classes={
            39:"bottle",
            41:"cup",
            43:"knife",
            44:"spoon",
            45:"bowl",
            62:"chair",
            63:"couch",
            67:"dining table",
            73:"laptop",
            74:"mouse",
            76:"keyboard",
            77:"cell phone"}  #using only 12 out of 80 available classes will be enough for now
    
    def process_frame(self,frame):  #starting tracking the frames
    
        frame_h,frame_w,_=frame.shape
        cnr_x_img,cnr_y_img=frame_w//2,frame_h//2
        ress=self.model.track(frame,persist=True,verbose=False)
        tracked_items=[]
        copied_frame=frame.copy()

        if ress[0].boxes is not None and len(ress[0].boxes)>0:
            boxes=ress[0].boxes
            ids=boxes.id.int().cpu().tolist() if boxes.id is not None else [0]*len(boxes)
            xyxy=boxes.xyxy.cpu().numpy()
            clss=boxes.cls.int().cpu().tolist()
            confs=boxes.conf.cpu().numpy()

            for idx,cls_id in enumerate(clss):
                if cls_id not in self.target_classes:
                    continue
                
                item_name=self.target_classes[cls_id]
                track_id=ids[idx]
                conf=confs[idx]